# 광고는 정말 구매 전환을 늘렸을까? — 마케팅 캠페인 A/B 테스트 분석

| 항목 | 내용 |
|---|---|
| 데이터 | [Marketing A/B Testing (Kaggle, faviovaz)](https://www.kaggle.com/datasets/faviovaz/marketing-ab-testing) |
| 내려받은 날짜 | 2026-11-08 |
| 분석자 | AI 데이터 인텔리전스 전문가과정 2기 박근아 |
| 질문 | 광고(ad)를 본 사용자가 공익광고(psa)를 본 사용자보다 더 많이 구매 전환했는가? |

**흐름 (6단계 고정)**
1 데이터 점검·전처리 → 2 질문·가설(결과 보기 전 고정) → 3 검정 선택 → 4 결과 분석(점추정·95% CI·p·효과크기) → 5 마케팅 의사결정(비용 민감도) → 6 보고서·한계

> 각 단계는 **지금 무엇을 하는가 → 왜 이 방법인가 → 무엇을 확인하고 어떻게 해석하는가** 순서로 적습니다.

## 0. 환경 준비

> 💡 **알아둘 것** 난수를 쓰지 않더라도 시드를 고정해 재현성을 보장합니다. 데이터는 `data/marketing_AB.csv`에 둡니다(원본 CSV는 GitHub에 올리지 않습니다).

In [2]:
%pip install statsmodels

   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ------ --------------------------------- 1.8/11.3 MB 10.0 MB/s eta 0:00:01
   ------------ --------------------------- 3.7/11.3 MB 9.9 MB/s eta 0:00:01
   ------------------- -------------------- 5.5/11.3 MB 8.8 MB/s eta 0:00:01
   ------------------------- -------------- 7.1/11.3 MB 8.7 MB/s eta 0:00:01
   -------------------------------- ------- 9.2/11.3 MB 8.9 MB/s eta 0:00:01
   ---------------------------------------  11.3/11.3 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 11.3/11.3 MB 8.5 MB/s  0:00:01

   ---------------------------------------- 0/6 [wrapt]
   ---------------------------------------- 0/6 [wrapt]
   ------ --------------------------------- 1/6 [patsy]
   ------ --------------------------------- 1/6 [patsy]
   ------ --------------------------------- 1/6 [patsy]
   ------ --------------------------------- 1/6 [patsy]
   ------------- -------------------------- 2/6 [narwh

In [3]:
import os
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import (proportions_ztest, confint_proportions_2indep,
                                          proportion_confint, proportion_effectsize)

np.random.seed(42)
plt.rcParams["font.family"] = {"Windows": "Malgun Gothic", "Darwin": "AppleGothic"}.get(platform.system(), "NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid")
os.makedirs("figures", exist_ok=True)
ALPHA = 0.05

In [4]:
import os
print(os.getcwd())
print(os.listdir("data"))

c:\Users\shini\Data Intelligence\AIFFEL_LMS_개인학습\05_통계적 검정과 인과추론 [종합과제]\AB-TEST-Marketing
['README.md']


In [5]:
print(raw.shape)
print(raw.info())
print(raw["test group"].value_counts(normalize=True).round(4))
print(raw.groupby("test group")["converted"].mean().round(4))
print(raw["total ads"].describe().round(1))

NameError: name 'raw' is not defined

## 1. 데이터 점검·전처리 (검정보다 먼저!)

**무엇을** 결측·중복·라벨·값의 범위·배정 비율을 확인하고, 제외한 데이터는 이유와 함께 기록합니다.
**왜** 오염된 데이터 위의 p값은 의미가 없기 때문입니다.

> ❓ **내가 물어볼 질문**
> - 행 수·컬럼·자료형·결측치는?
> - 같은 사용자가 두 번 이상 나오나?
> - 그룹 라벨은 ad / psa 두 개뿐인가? 값의 범위는 상식적인가?
> - 배정 비율은 **원래 계획**대로인가? 원래 계획을 알 수 있나?

In [ ]:
raw = pd.read_csv("data/marketing_AB.csv")
print("원본 크기:", raw.shape)
print(raw.dtypes)
raw.head()

In [ ]:
# 컬럼 이름 정리(공백 → 밑줄), 저장용 인덱스 컬럼 제거
df = raw.drop(columns=[c for c in raw.columns if c.startswith("Unnamed")]).copy()
df.columns = [c.strip().replace(" ", "_") for c in df.columns]
print(df.columns.tolist())

print("\n[결측치]");          print(df.isna().sum())
print("\n[그룹 라벨]");       print(df["test_group"].value_counts())
print("\n[converted 값]");    print(df["converted"].value_counts())
print("\n[중복 사용자 ID 수]", df["user_id"].duplicated().sum())
print("[값 범위] total_ads 최소", df["total_ads"].min(), "/ most_ads_hour", df["most_ads_hour"].min(), "~", df["most_ads_hour"].max())
print("[요일 값]", sorted(df["most_ads_day"].unique()))

In [ ]:
# 정제: 중복 사용자 제외(있다면), converted를 0/1로
n_before = len(df)
df = df.drop_duplicates(subset="user_id", keep=False)   # 두 번 이상 나온 사용자는 배정이 모호하므로 모두 제외
df = df[df["test_group"].isin(["ad", "psa"])]           # 예상 밖 라벨 제외
df["converted"] = df["converted"].astype(int)
n_after = len(df)
print(f"정제 전 {n_before:,}행 → 정제 후 {n_after:,}행 (제외 {n_before - n_after:,}행)")

> ✍️ **메모** 제외 __행, 이유: ____ (제외가 없다면 "점검 결과 제외 대상 없음"이라고 적습니다.)

### 1-1. 배정 비율과 SRM

**무엇을** 두 그룹의 실제 비율을 확인합니다.
**왜** 배정이 계획과 다르면(SRM) 배정 로직이나 로깅의 오류를 의심해야 하기 때문입니다.
**어떻게 해석** SRM 검정은 **p가 커야 정상**입니다. 그런데 SRM은 **원래 계획한 비율과 비교**해야만 판단할 수 있습니다.

> ⚠️ 이 데이터는 50:50 실험이 아니라 대부분에게 광고를 보여 준 설계입니다. **불균형 자체는 오류가 아닙니다.** 원래 계획한 비율을 알려 주는 문서가 없으므로, 관측된 비율로 '기대 비율'을 만들어 검정하는 것은 순환논리입니다. 따라서 **SRM 판정은 유보**하고 한계에 기록합니다.

In [ ]:
n_ad, n_psa = (df["test_group"] == "ad").sum(), (df["test_group"] == "psa").sum()
total = n_ad + n_psa
print(f"ad {n_ad:,} ({n_ad/total:.1%})  |  psa {n_psa:,} ({n_psa/total:.1%})")

# 참고: 50:50을 기대 비율로 두면? → 이 실험의 계획과 맞지 않으므로 판정 근거로 쓰지 않는다
print("참고) 50:50 기대 시 SRM p =", stats.chisquare([n_ad, n_psa], [total/2, total/2]).pvalue)

> ✍️ **메모** 관측 비율 ad __% : psa __%. 원래 배정 계획 정보가 없어 **SRM 판정 유보**. 무작위 배정이 적절했다는 **전제 하에서** 이후 결과를 인과적으로 해석한다.

### 1-2. 처치 이후 변수 확인 (`total_ads`, `most_ads_day`, `most_ads_hour`)

**무엇을** 광고 노출 수와 노출 요일·시간의 분포를 봅니다.
**왜** 이 변수들은 **배정 이후의 행동**에서 생긴 값입니다. 결과와 함께 움직일 수 있어 **통제 변수로 넣으면 안 되고**(나쁜 통제), 특정 요일·시간이 구매의 원인이라고 결론 내려서도 안 됩니다.
**어떻게** 주 분석은 **배정된 그룹 그대로** 비교합니다(ITT). 극단값은 실제 헤비 유저일 수 있어 **삭제하지 않고 유지**합니다.

In [ ]:
print(df.groupby("test_group")["total_ads"].describe().round(1))
fig, ax = plt.subplots(figsize=(8, 3.2))
sns.boxplot(data=df, x="total_ads", y="test_group", ax=ax, showfliers=False)
ax.set_title("그룹별 광고 노출 수 (극단값 표시 생략, 데이터는 유지)")
plt.tight_layout(); plt.show()

## 2. 질문과 가설 — 결과를 보기 전에 고정

> ⚠️ 여기서 정한 가설·검정·유의수준·**의사결정 규칙**은 결과를 본 뒤 바꾸지 않습니다(바꾸면 p-해킹).

- **질문:** 광고가 공익광고 대비 구매 전환율을 높였는가?
- **주 지표:** 구매 전환율 (`converted`, 0/1)
- **H₀:** 광고군과 공익광고군의 전환율은 같다. (p_ad = p_psa)
- **H₁:** 두 그룹의 전환율은 다르다. (p_ad ≠ p_psa)
- **유의수준 α = 0.05, 양측 검정.** 광고가 오히려 전환을 낮출 가능성도 열어 둔다.

**의사결정 규칙(미리 정함)** 5단계에서 계산할 **손익분기 상승폭**을 기준으로,
- 95% 신뢰구간 **하한**이 손익분기 이상 → **출시**
- 점추정은 손익분기 이상이지만 하한은 미만 → **추가 수집**
- 점추정이 손익분기 미만, 또는 차이가 유의하지 않음 → **보류**

## 3. 검정 고르기 ⭐

> ❓ **내가 물어볼 질문 세 가지** ① 지표가 비율인가 연속형인가? ② 집단이 2개인가? ③ (연속형이면) 정규성·등분산은?

**나는 두 비율 z-검정(= 2×2 카이제곱 독립성 검정)을 골랐다.** 왜냐하면 주 지표 `converted`가 **구매했다/안 했다의 비율**이고, 비교 집단이 **ad와 psa 두 개**이기 때문이다. 비율 지표라 정규성·등분산 가정은 필요 없고, 대신 각 칸의 도수가 충분해야 하는데(전환·비전환 모두 10 이상) 아래에서 확인한다. z-검정은 차이의 **방향과 신뢰구간**을 바로 주고, 카이제곱과는 χ² = z²로 같은 결론이므로 효과크기(Cramér's V) 계산에 함께 쓴다. 그룹 크기가 크게 다르지만, 두 비율 검정은 각 그룹의 표본 수를 따로 반영하므로 불균형 자체가 검정을 무효로 만들지는 않는다. 다만 불확실성은 주로 **작은 쪽(psa)** 에서 온다.

In [ ]:
ct = pd.crosstab(df["test_group"], df["converted"])
print(ct)
print("모든 칸 ≥ 10 ?", (ct.values >= 10).all())

## 4. 결과 분석 — 점추정 · 95% 신뢰구간 · p값 · 효과크기

**무엇을** 차이의 크기, 불확실성, 유의성, 효과크기를 함께 계산합니다.
**왜** p값은 "우연인가"만 말해 줍니다. "얼마나 큰가"는 점추정·신뢰구간·효과크기가 말해 줍니다.
**어떻게 해석** 비율 지표는 **절대 차이(%p)와 상대 차이(배수)를 둘 다** 적습니다. 상대 차이만 쓰면 과장되고, 절대 차이만 쓰면 작아 보입니다.

In [ ]:
c_ad  = df.loc[df.test_group == "ad",  "converted"].sum()
c_psa = df.loc[df.test_group == "psa", "converted"].sum()
p_ad, p_psa = c_ad / n_ad, c_psa / n_psa
lift = p_ad - p_psa

z, p_z = proportions_ztest([c_ad, c_psa], [n_ad, n_psa])
lo, hi = confint_proportions_2indep(c_ad, n_ad, c_psa, n_psa, method="wald")
chi2, p_chi, dof, _ = stats.chi2_contingency(ct, correction=False)
cramers_v = np.sqrt(chi2 / ct.values.sum())
cohens_h = proportion_effectsize(p_ad, p_psa)

print(f"전환율        ad {p_ad:.3%}  vs  psa {p_psa:.3%}")
print(f"① 절대 차이   {lift*100:+.2f}%p   95% CI [{lo*100:+.2f}, {hi*100:+.2f}]%p")
print(f"   상대 차이   {p_ad / p_psa:.2f}배  ({(p_ad / p_psa - 1)*100:+.0f}%)")
print(f"② 검정        z = {z:.2f}, p = {p_z:.2e}   |   χ²({dof}) = {chi2:.1f}")
print(f"③ 효과크기    Cramér's V = {cramers_v:.3f},  Cohen's h = {cohens_h:.3f}")

In [ ]:
ci_ad  = proportion_confint(c_ad,  n_ad,  method="wilson")
ci_psa = proportion_confint(c_psa, n_psa, method="wilson")
fig, ax = plt.subplots(figsize=(7, 3.2))
for i, (r, ci) in enumerate([(p_psa, ci_psa), (p_ad, ci_ad)]):
    ax.errorbar(r*100, i, xerr=[[(r-ci[0])*100], [(ci[1]-r)*100]], fmt="o", capsize=6)
ax.set_yticks([0, 1]); ax.set_yticklabels(["psa (대조)", "ad (광고)"])
ax.set_xlabel("구매 전환율 (%) · 95% 신뢰구간"); ax.set_title("그룹별 구매 전환율")
plt.tight_layout(); plt.savefig("figures/conversion_ci.png", dpi=150); plt.show()

> ✍️ **보고 문장(출력 숫자로 채우기)** 광고군의 전환율이 [더 높았다/낮았다] (__% vs __%). __%p (95% CI [__, __]), 상대 __배, z = __, p = __, Cramér's V = __.
>
> ❓ **내가 물어볼 질문** 차이가 유의한가? 그리고 **효과크기는 큰가 작은가?** 표본이 크면 작은 차이도 유의해진다. 그래서 다음 단계에서 **돈이 되는지**를 따로 본다.

## 5. 마케팅 의사결정 — 비용 민감도 분석 (한 걸음 더 · D)

**무엇을** 전환 1건의 이익과 광고 노출 비용을 **가정**하고, 손익분기 상승폭과 추가 이익을 계산합니다.
**왜** "유의하다"는 "우연이 아니다"일 뿐 "돈이 된다"가 아닙니다. 실제 광고비·이익 데이터가 없으므로 ROI를 확정할 수 없고, 대신 **어떤 조건에서 광고가 유리한지**를 보여 줍니다.
**어떻게 해석** 2단계에서 정한 의사결정 규칙에 그대로 대입합니다.

> ❓ **내가 물어볼 질문** 전환 1건의 이익은 얼마로 가정할까? 노출 1회 비용은? 신뢰구간 **하한**에서도 이익인가? 가정을 바꾸면 결론이 뒤집히는가?

In [ ]:
# ── 기준 가정 (근거를 보고서에 적고, 아래 민감도 표에서 바꿔 봄) ──
VALUE_PER_CONV = 30.0     # 전환 1건의 이익(달러)
COST_PER_AD    = 0.002    # 광고 노출 1회 비용(달러) = CPM 2달러

total_ads_ad   = df.loc[df.test_group == "ad", "total_ads"].sum()
ad_cost        = total_ads_ad * COST_PER_AD
breakeven_lift = ad_cost / (VALUE_PER_CONV * n_ad)      # 이만큼 전환율이 올라야 본전

def profit(l, value=VALUE_PER_CONV, cost=COST_PER_AD):
    return l * n_ad * value - total_ads_ad * cost

print(f"광고 비용 합계      ${ad_cost:,.0f}")
print(f"손익분기 상승폭     {breakeven_lift*100:.3f}%p\n")
for name, l in [("CI 하한", lo), ("점추정", lift), ("CI 상한", hi)]:
    print(f"{name:6s} {l*100:+.3f}%p → 추가 이익 ${profit(l):,.0f}")

In [ ]:
# 민감도: 전환 가치 × 노출 비용 조합별 추가 이익 (점추정 기준, 달러)
values = [10, 30, 60]
cpms   = [0.001, 0.002, 0.005]
grid = pd.DataFrame({f"CPM ${c*1000:.0f}": [profit(lift, v, c) for v in values] for c in cpms},
                    index=[f"전환가치 ${v}" for v in values]).round(0)
grid

In [ ]:
# 2단계에서 미리 정한 의사결정 규칙 적용
significant = p_z < ALPHA
if significant and lo >= breakeven_lift:
    decision = "출시"
elif significant and lift >= breakeven_lift:
    decision = "추가 수집"
else:
    decision = "보류"
print(f"유의성: {'유의' if significant else '유의하지 않음'} | 손익분기 {breakeven_lift*100:.3f}%p | "
      f"CI 하한 {lo*100:.3f}%p | 점추정 {lift*100:.3f}%p")
print("→ 기준 가정 하 의사결정:", decision)

## 6. 보고서 요약 (README와 같은 흐름)

1. **질문과 가설** — 광고가 공익광고 대비 구매 전환율을 바꾸는가? H₀: 같다 / H₁: 다르다, α = 0.05, 양측.
2. **데이터 점검** — 정제 전 __행 → 후 __행. 결측 __, 중복 __. 배정 ad __% : psa __%, 원래 계획 정보가 없어 SRM 판정 유보.
3. **검정 선택** — 비율 지표 · 2집단 → 두 비율 z-검정(= 카이제곱). 모든 칸 도수 충분.
4. **결과** — __%p (95% CI [__, __]), 상대 __배, p = __, Cramér's V = __.
5. **의사결정** — 기준 가정(전환 $__, CPM $__) 하 **____**. 민감도: ____ 조건에서는 결론이 바뀜.
6. **한계** — 무작위 배정 절차를 독립적으로 검증할 수 없음(적절했다는 전제), SRM 판정 유보, 대조군이 작음, 비용·이익은 가정값, 단기 전환만 측정, 수집 시기·지역 정보 없음, 노출 요일·시간·횟수는 처치 이후 변수라 원인으로 해석하지 않음.